# 코랩으로 형태 분석 말뭉치에서 명사 뽑기

**인공지능과 인문학 · 9주차 미니 실습**
송영숙 · 단국대학교 문과대학 · 2026학년도 1학기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/songys/me/blob/main/dankook-2026-week09/%EB%AA%85%EC%82%AC%EB%BD%91%EA%B8%B0_%EB%AF%B8%EB%8B%88%EC%8B%A4%EC%8A%B5.ipynb)

---

## 이 실습에서 하는 일
국립국어원 형태 분석 말뭉치(JSON)에서 **명사만 골라 뽑고**, **가장 많이 나온 단어 순으로 줄을 세웁니다**.

코드는 5~10줄짜리 스니펫 4개로 끝납니다. 인문학 학생이 처음 보아도 복사·붙여넣기만 하면 결과가 나옵니다.

## 준비물
- **샘플 파일** — `sample_morph_corpus.json` (43KB, 문장 10개)
  - e-class에서 받음
  - 출처: 국립국어원 형태 분석 말뭉치 버전 1.0(SXMP1902008031.json)에서 처음 두 문서·다섯 문장씩 발췌
- **인터넷 + 구글 계정**

## 0. 샘플 파일 코랩에 올리기

아래 셀을 실행(`Shift + Enter`)하면 파일 선택 창이 뜹니다. `sample_morph_corpus.json`을 골라 업로드합니다. 1~2초면 끝납니다.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 1. 5줄로 명사만 뽑기

JSON 구조가 서랍 안에 또 서랍이 있는 모양이라, 안쪽 서랍까지 들어가는 `for` 세 번을 씁니다.

- `data['document']` — 말뭉치 안의 여러 문서
- `doc['sentence']` — 한 문서 안의 여러 문장
- `sent['morpheme']` — 한 문장을 잘게 쪼갠 형태소들
- `m['label'] == 'NNG'` — 그중 일반명사(NNG)만 골라
- `m['form']` — 형태(글자)를 리스트에 모음

In [ ]:
import json

with open('sample_morph_corpus.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

nouns = []
for doc in data['document']:
    for sent in doc['sentence']:
        for m in sent['morpheme']:
            if m['label'] == 'NNG':
                nouns.append(m['form'])

print(f"전체 일반명사 개수: {len(nouns)}")
print(f"앞 20개: {nouns[:20]}")

### 예상 결과
```
전체 일반명사 개수: 27
앞 20개: ['요즘', '날씨', '라테', '한잔', '라테', '위', '다양', '라테', '아트', '구경', '시행', '제도', '체포', '우선', '주의', '제도', '부분', '주', '가정', '폭력']
```

## 2. 가장 많이 나온 명사 줄 세우기

`Counter`는 파이썬 기본에 들어 있는 도구입니다. 따로 설치할 필요 없습니다.

In [ ]:
from collections import Counter

counter = Counter(nouns)
print("=== 가장 많이 나온 일반명사 10개 ===")
for word, count in counter.most_common(10):
    print(f"  {word}: {count}회")

### 예상 결과
```
=== 가장 많이 나온 일반명사 10개 ===
  라테: 3회
  제도: 2회
  가정: 2회
  폭력: 2회
  요즘: 1회
  ...
```

### 한 줄 해석
- 1위는 **'라테' 3회** — 첫 문서가 카페 이야기인 듯
- '제도', '가정', '폭력'이 2회씩 — 두 번째 문서는 사회 주제로 보임
- **빈도만 봐도 두 문서의 주제가 갈라짐**

## 3. 다른 품사도 — 동사·형용사·고유명사

**`label`만 바꾸면** 다른 품사도 같은 방식으로 뽑을 수 있습니다.

In [ ]:
from collections import Counter

labels = {
    'NNG': '일반명사',
    'NNP': '고유명사',
    'VV':  '동사',
    'VA':  '형용사',
}

for label, label_kr in labels.items():
    items = []
    for doc in data['document']:
        for sent in doc['sentence']:
            for m in sent['morpheme']:
                if m['label'] == label:
                    items.append(m['form'])
    top5 = Counter(items).most_common(5)
    print(f"\n[{label_kr} {label}] 총 {len(items)}개")
    for word, count in top5:
        print(f"  {word}: {count}회")

### 한 줄 해석
- **고유명사 '미국' 2회** — 두 번째 문서가 미국 제도를 다룸
- **형용사 '춥다·따뜻하다'** — 첫 문서가 날씨·온도 이야기
- 라벨 한 글자만 바꿔 다양한 단면을 본 셈

## 4. (선택) 한 걸음 더 — 워드클라우드

명사 빈도를 그림으로 그려 봅니다. 한국어 폰트 설치가 필요합니다.

In [ ]:
# 한국어 폰트 설치 (1회만)
!apt-get install fonts-nanum -y > /dev/null 2>&1
!pip install wordcloud > /dev/null 2>&1

from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter

counter = Counter(nouns)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

wc = WordCloud(
    font_path=font_path,
    background_color='white',
    width=800, height=400,
    max_words=50,
).generate_from_frequencies(counter)

plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.show()

### 무엇이 보이나
- '라테'가 가장 크게 — 빈도가 높을수록 글자가 큼
- 작은 데이터(문장 10개)이라 단어가 적게 보임 — 큰 데이터일수록 풍부한 그림

본인의 분석 데이터를 같은 방식으로 워드클라우드로 만들 수 있습니다. 11주차에 본격적으로 다룹니다.

## 5. 막혔을 때 — 바이브 코딩으로 묻기

코드가 안 돌아가거나 결과가 이상하면, **AI에게 친구처럼 물어봅니다**.

### 질문 템플릿 (그대로 복사해서 채우기)

> "나는 구글 코랩에서 이 코드를 실행하려고 했어:
> 
> ```python
> (코드 붙여넣기)
> ```
> 
> 이런 에러가 났어:
> 
> ```
> (에러 메시지 마지막 줄을 그대로 붙여넣기)
> ```
> 
> 내가 원했던 건 형태 분석 말뭉치에서 명사만 뽑는 일인데, 어떻게 고치면 좋을까?"

이렇게 물으면 ChatGPT·Claude·Gemini가 어디가 잘못됐는지 짚어 줍니다. 답이 두 번까지 안 통하면 다른 키워드로 검색하거나 옆 사람에게 손을 드세요.

## 부록 A. 한국어 형태소 라벨 표

이 실습에서 만난 라벨과 자주 쓰는 추가 라벨입니다.

### 명사 계열
| 라벨 | 뜻 | 예 |
|------|----|---|
| NNG | 일반명사 | 라테, 제도, 날씨 |
| NNP | 고유명사 | 미국, 송영숙, 서울 |
| NNB | 의존명사 | 것, 수, 만 |
| NR  | 수사 | 하나, 천구백팔십 |

### 용언 계열
| 라벨 | 뜻 | 예 |
|------|----|---|
| VV | 동사 | 찾(다), 그리(다) |
| VA | 형용사 | 춥(다), 따뜻하(다) |
| VX | 보조용언 | 보(다), 주(다) |
| VCP | 긍정지정사 | 이(다) |

### 조사
| 라벨 | 뜻 | 예 |
|------|----|---|
| JKS | 주격 조사 | 이/가 |
| JKB | 부사격 조사 | 에, 로, 처럼 |
| JKO | 목적격 조사 | 을/를 |
| JKG | 관형격 조사 | 의 |
| JX  | 보조사 | 는, 도, 만 |

### 어미
| 라벨 | 뜻 | 예 |
|------|----|---|
| EP | 선어말 어미 | -었-, -겠- |
| EF | 종결 어미 | -다, -아요 |
| EC | 연결 어미 | -고, -면 |
| ETM | 관형형 어미 | -ㄴ, -ㄹ |

**TIP**: 분석에서 가장 자주 쓰는 라벨은 명사(NNG, NNP), 동사(VV), 형용사(VA). 이 네 개만 잘 알면 대부분의 분석을 시작할 수 있습니다.

## 부록 B. 이 실습에서 쓴 데이터

- **이름**: 국립국어원 형태 분석 말뭉치 (버전 1.0)
- **원본 파일**: SXMP1902008031.json
- **발췌 범위**: 처음 2개 문서, 각 5문장 (학습 목적의 짧은 발췌)
- **샘플 파일 크기**: 43KB
- **출처**: 국립국어원 모두의 말뭉치 — corpus.korean.go.kr
- **활용 시 주의**: 원본 데이터의 라이선스를 사용 직전 다시 확인할 것 (학술용 외 재배포에 주의)

---

## 정리
- 코랩에서 5줄로 명사만 뽑기 가능
- 라벨만 바꾸면 동사·형용사·고유명사로 확장
- 빈도 결과만 봐도 두 문서의 주제가 갈라지는 것을 확인
- 11주차부터 본인 분석 데이터에 같은 방식을 적용
- 에러는 두렵지 않다 — 메시지를 그대로 AI에게 물으면 풀림